# Charts checking initial labelled data

In [1]:
import pandas as pd
import requests
import altair as alt
import json

from discovery_child_development import PROJECT_DIR
from discovery_child_development.getters import openalex, patents
from discovery_child_development.utils import plotting_utils as pu
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import altair_save_utils
from discovery_child_development.utils import openalex_utils

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

2024-04-29 18:24:18,802 - botocore.credentials - INFO - Found credentials in environment variables.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-04-29 18:24:22,577 - datasets - INFO - PyTorch version 2.1.2 available.


In [2]:
AltairSaver = altair_save_utils.AltairSaver()

2024-04-29 18:24:23,045 - root - INFO - Downloading chromedriver (124.0.6367.91)...
CHROME >= 115, using mac-arm64 as architecture identifier


## Set up

### Global variables

In [3]:
openalex_label = "Publications"
patents_label = "Patents"

### Load in the labelled data

In [67]:
# Load labelled data
data_df = (
    pd.read_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_filtered.csv')
    .assign(id = lambda df: df['id'].apply(lambda x: x.split('/')[-1]))
)

gtr_data_df = (
    pd.read_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_gtr_filtered.csv')
    .assign(source="gtr")
    .assign(year=lambda df: df['start'].apply(lambda x: int(x.split('-')[0])))
    .drop(columns=['start', 'end'])
)

# gtr_metadata_df = gtr_data_df[['id','start']].copy()
# gtr_data_df = gtr_data_df[['id', 'text', 'source', 'topics', 'year']]
# data_df = pd.concat([data_df, gtr_data_df], ignore_index=True)

In [60]:
gtr_data_df.head(1)

,id,text,topics,source,year
0,F8886352-F22C-4E4C-AFAC-14A7D8CE0FF3,Strengthening health system promotion of mater...,health,gtr,2015


In [68]:
# Detection and management
detection_df = (
    pd.read_csv(ENRICHED_DATA_DIR / "openalex_patents_detection_labels.csv")
    .assign(id = lambda df: df['id'].apply(lambda x: x.split('/')[-1]))
)

data_df = (
    data_df
    .merge(detection_df[['id', 'Detection', 'Detection_', 'Management', 'Management_']], on='id', how='left')
    .drop_duplicates(subset='id', keep='first')
    .drop_duplicates(subset='text', keep='first')
    .rename(columns={'source': 'Dataset'})
    .replace({'Dataset': {"openalex": openalex_label, "patents": patents_label}})    
)

In [69]:
print(len(data_df))

49805


In [70]:
data_df.tail()

,id,text,Dataset,topics,Detection,Detection_,Management,Management_
51229,W4380048305,Trilingual families' language strategies: pote...,Publications,communication,0.401767,0,0.797585,1
51230,W4385650553,The Effects of Vitamin D Supplementation on Re...,Publications,"rct, nutrition, health",0.992909,1,0.000782,0
51231,W4367694030,A Study to Assess the Effectiveness of Video A...,Publications,"social_services, protection",0.002119,0,0.996927,1
51232,W4380886404,Vaginal Bleeding In Prepubertal Girls-A Case S...,Publications,NaN,0.576360,1,0.154030,0
51233,W4386132309,The construction and preliminary validation of...,Publications,NaN,0.744173,1,0.033589,0


### Load in topic information

In [7]:
# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

topics_df = []
for topic in topics_dict:
    topics_df.append({
        'topic': topic,
        'type': topics_dict[topic]['type'],
        'name': topics_dict[topic]['name'],
        # 'description': topics_dict[topic]['description']
    })
topics_df = (
    pd.DataFrame(topics_df)
    .sort_values(['type', 'topic',])
    .reset_index(drop=True)
    .replace('Family and home', 'Parenting')
    .rename(columns={'type': 'Type'})
)

38


In [8]:
topics_df

,topic,Type,name
0,arts,Area of learning,Expressive arts and design
1,communication,Area of learning,Communication and language
2,emotional,Area of learning,Personal social emotional
3,literacy,Area of learning,Literacy
4,mathematics,Area of learning,Mathematics
5,physical,Area of learning,Physical
6,cognitive,Development,Cognitive development
7,infancy,Development,Infancy
8,prenatal,Development,Prenatal
9,send,Development,Special needs


### Load in metadata

In [9]:
openalex_metadata_df = openalex.get_concepts_metadata()
patents_metadata_df = patents.get_patents_from_s3()

In [10]:
openalex_metadata_df.head(1)

,openalex_id,title,year,extracted_date_time,concept_id,wikidata,display_name,level,score
0,https://openalex.org/W2146349261,The impact of pretend play on children's devel...,2013,2024-02-20 12:07:12,https://openalex.org/C183030095,https://www.wikidata.org/wiki/Q255894,Equifinality,2,0.788561


In [51]:
import ast
# data from 03_geo_analysis notebooks
openalex_extra_metadata_df = (
    pd.read_csv(ENRICHED_DATA_DIR / 'pubs_metadata_df.csv')
    .assign(country_code = lambda df: df['country_code'].apply(lambda x: ast.literal_eval(x)))
)


In [52]:
openalex_extra_metadata_df.head(1)

,id,country_code,cited_by_count
0,W2134744704,[US],610


### Join labelled data with metadata

In [82]:
patents_df = (
    data_df
    .query('Dataset == "Patents"')
    .merge(
        (
            patents_metadata_df
            .rename(columns={'publication_number': 'id'})
            # .assign(year=lambda df: df['filing_date'].apply(lambda x: int(str(x)[0:4])))
            .assign(year=lambda df: df['publication_date'].apply(lambda x: int(str(x)[0:4])))
            .query('year >= 2013')
        )[['id', 'year', 'country_code']],
        on='id',
    )
)

openalex_df = (
    data_df
    .query('Dataset == "Publications"')
    .merge(
        (
            openalex_metadata_df
            .rename(columns={"openalex_id": "id"})
            .assign(id=lambda df: df['id'].apply(lambda x: x.split("/")[-1]))
            .drop_duplicates(subset=['id'])
        )[['id', 'year']]
        ,
        how='left',
        on='id'
    )
    .merge(
        openalex_extra_metadata_df,
        how='left',
        on='id'
    )
)

relevant_df = pd.concat([patents_df, openalex_df, gtr_data_df.assign(Dataset='UKRI')], ignore_index=True)

In [83]:
print(len(relevant_df))

50898


In [84]:
relevant_df.tail()

,id,text,Dataset,topics,Detection,Detection_,Management,Management_,year,country_code,cited_by_count,source
50893,66CE94D7-5CE8-462A-830C-E9E11EA2FE3D,VACCINE: Development of Novel BRSV Pre-Fusion ...,UKRI,infancy,NaN,NaN,NaN,NaN,2017,NaN,NaN,gtr
50894,DCE50EED-4979-461F-809E-F51840CDC9E7,Understanding and Enhancing Repair of the Ear ...,UKRI,NaN,NaN,NaN,NaN,NaN,2019,NaN,NaN,gtr
50895,E7D450F6-FE6C-4343-913A-E1DCEAB55224,Language Development and Disorder in Children ...,UKRI,"send, infancy, communication",NaN,NaN,NaN,NaN,2019,NaN,NaN,gtr
50896,79032239-176B-4B40-A2A7-EBCF76AA12CF,Developmental Differences in Cognitive Control...,UKRI,"cognitive, neuroscience, games",NaN,NaN,NaN,NaN,2017,NaN,NaN,gtr
50897,E0164690-9B2C-4C58-8764-FAB8FEABA520,The Smoke-free Homes Innovation Network (SHINE...,UKRI,"social_media, infancy, health, protection",NaN,NaN,NaN,NaN,2021,NaN,NaN,gtr


In [85]:
(
    relevant_df
    # .assign(country_code = lambda df: df.country_code.apply(lambda x: ",".join(x) if isinstance(x, list) else x))
    .to_csv(ENRICHED_DATA_DIR / 'relevant_labelled_df_for_atlas.csv', index=False)
)

### Prepare topic dataset

In [75]:
exploded_relevant_df = (
    relevant_df
    .dropna(subset=['topics'])
    .assign(topics_list=lambda df: df.topics.apply(lambda x: [x.strip() for x in x.split(",")]))
    .explode('topics_list')
    .drop_duplicates(subset=['id', 'topics_list'])
)

topic_counts = (
    exploded_relevant_df
    .groupby("topics_list")
    .agg(counts=("id", "count"))
    .reset_index()
)

# topics_df = (
#     topics_df_
#     .merge(topic_counts, left_on='topic', right_on='topics_list', how='left')
#     .fillna(0)
#     .sort_values(['type', 'counts'], ascending=[True, False])
#     # replace Family and home with Parenting
# )

In [25]:
topics_df


,topic,Type,name
0,arts,Area of learning,Expressive arts and design
1,communication,Area of learning,Communication and language
2,emotional,Area of learning,Personal social emotional
3,literacy,Area of learning,Literacy
4,mathematics,Area of learning,Mathematics
5,physical,Area of learning,Physical
6,cognitive,Development,Cognitive development
7,infancy,Development,Infancy
8,prenatal,Development,Prenatal
9,send,Development,Special needs


## Publications and patents data

### Number of documents per dataset

In [87]:
# Number of papers and patents about detection and management
fraction_detection_df = (
    relevant_df
    .groupby(["Dataset"])
    .agg(
        n=('id', 'count'),
        n_detection=('Detection_', 'sum'),
        n_management=('Management_', 'sum'),
    )
    .assign(
        fraction_detection = lambda df: df['n_detection'] / df['n'],
        fraction_management = lambda df: df['n_management'] / df['n'],
    )
    .reset_index()
)
# Turn 
fraction_detection_df

,Dataset,n,n_detection,n_management,fraction_detection,fraction_management
0,Patents,11534,2721.0,8582.0,0.235911,0.744061
1,Publications,38271,13159.0,26396.0,0.343837,0.689713
2,UKRI,1093,0.0,0.0,0.000000,0.000000


In [88]:
# Chart with number of documents per dataset
fig = (
    alt.Chart(fraction_detection_df)
    .mark_bar()
    .encode(
        x=alt.X('n:Q', title=''),
        y=alt.Y('Dataset:N', title='', sort='-x'),
        color=alt.Color('Dataset:N', title=None, legend=None),
        tooltip=['Dataset', 'n',]
    )
)
fig = pu.configure_titles(pu.configure_plots(fig), "Number of documents per dataset")

In [89]:
fig

/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/altair/utils/core.py:317: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for col_name, dtype in df.dtypes.iteritems():


alt.Chart(...)

In [90]:
n_publications = fraction_detection_df.loc[fraction_detection_df.Dataset=="Publications", "n"].iloc[0]
n_patents = fraction_detection_df.loc[fraction_detection_df.Dataset=="Patents", "n"].iloc[0]
print(f"Number of publications: {n_publications}")
print(f"Number of patents: {n_patents}")
print(f"Ratio of publications to patents: {n_publications/n_patents:.2f}")

Number of publications: 38271
Number of patents: 11534
Ratio of patents to publications: 3.32


In [ ]:
savename = 'number_of_documents_per_dataset'
AltairSaver.save(fig, savename)

### Number of documents over time

#### Get baseline stats for normalisation

In [91]:
# Publications baseline
openalex_baseline = (
    openalex_utils.get_publications_count_per_year(start_year=2013, end_year=2024)
    .assign(Dataset="Publications")
)


In [ ]:
openalex_baseline.head()

In [92]:
# Patents baseline
patents_baseline = pd.DataFrame(json.load(open("total_patents.json", "r")))
patents_baseline = (
    patents_baseline
    .assign(Dataset="Patents")
    .rename(columns={"total_publications": "total_counts", "publication_year": "year"})
    .astype({"year": int, "total_counts": int})
)

# Combined baseline
baseline_df = pd.concat([openalex_baseline, patents_baseline], ignore_index=True)

In [93]:
au.ts_magnitude_growth_(
    ts_df = (
        openalex_baseline
        .query("year < 2024")
    ),
    year_start = 2019,
    year_end = 2023   
)

/Users/karlis.kanders/Documents/code/discovery_child_development/discovery_child_development/utils/analysis_utils.py:716: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  magnitude = time_series.set_index("year").loc[year_start:year_end, :].mean()
/Users/karlis.kanders/Documents/code/discovery_child_development/discovery_child_development/utils/analysis_utils.py:693: FutureWarning: Dropping of nuisance columns in rolling operations is deprecated; in a future version this will raise TypeError. Select only valid columns before calling the operation. Dropped columns were Index(['Dataset'], dtype='object')
  df_ma = timeseries_df.rolling(window, min_periods=1).mean().drop("year", axis=1)


,magnitude,growth
total_counts,9855088.2,-3.874384


In [94]:
au.ts_magnitude_growth_(
    ts_df = (
        patents_baseline
        .query("year < 2024")
    ),
    year_start = 2019,
    year_end = 2023   
)

/Users/karlis.kanders/Documents/code/discovery_child_development/discovery_child_development/utils/analysis_utils.py:716: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  magnitude = time_series.set_index("year").loc[year_start:year_end, :].mean()
/Users/karlis.kanders/Documents/code/discovery_child_development/discovery_child_development/utils/analysis_utils.py:693: FutureWarning: Dropping of nuisance columns in rolling operations is deprecated; in a future version this will raise TypeError. Select only valid columns before calling the operation. Dropped columns were Index(['Dataset'], dtype='object')
  df_ma = timeseries_df.rolling(window, min_periods=1).mean().drop("year", axis=1)


,magnitude,growth
total_counts,7572871.2,31.192759


### Number of documents per year

In [95]:
counts_df = (
    relevant_df
    # .query("country_code != 'CN'")
    .groupby(['Dataset', 'year'])
    .agg(counts=('id', 'count'))
    .reset_index()
    .merge(
        baseline_df,
        on=['year', 'Dataset'],
        how='left'
    )    
    .assign(
        # counts = lambda df: df['counts'] / 1e6,
        # total_counts = lambda df: df['total_counts'] / 1e6,
        fraction = lambda df: df['counts'] / df['total_counts'],
        percentage = lambda df: df['fraction'] * 100,
    )
)

# Normalise counts in each source by year 2013
counts_normalised_df = (
    counts_df
    .merge(
        counts_df.query('year == 2013').rename(columns={'fraction': 'fraction_2013'})[['Dataset', 'fraction_2013']],
        on='Dataset',
        how='left'
    )
    .assign(
        normalised_fraction = lambda df: df['fraction'] / df['fraction_2013']
    )
)
    

In [98]:
counts_df

,Dataset,year,counts,total_counts,fraction,percentage
0,Patents,2013,484,4271238.0,0.000113,0.011332
1,Patents,2014,510,4459288.0,0.000114,0.011437
2,Patents,2015,839,4856237.0,0.000173,0.017277
3,Patents,2016,915,5152383.0,0.000178,0.017759
4,Patents,2017,1005,5566320.0,0.000181,0.018055
5,Patents,2018,1219,6297186.0,0.000194,0.019358
6,Patents,2019,1283,6472541.0,0.000198,0.019822
7,Patents,2020,1402,7336249.0,0.000191,0.019111
8,Patents,2021,1458,8455856.0,0.000172,0.017242
9,Patents,2022,1303,8122074.0,0.000160,0.016043


In [96]:
import importlib
importlib.reload(pu);

In [97]:
fig_title = "Documents by year"
fig_subtitle = ""

fig = pu.ts_smooth(
    (
        counts_normalised_df
        .query('year < 2024')
    ),
    ["Publications", "Patents"],
    variable= "counts",
    variable_title = "",
    category_column = "Dataset",
    width = 300,
    height = 150,
    legend_orient='right',
)
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle)
fig

/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/altair/utils/core.py:317: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for col_name, dtype in df.dtypes.iteritems():


alt.Chart(...)

In [ ]:
n_2013 = counts_normalised_df.query("Dataset == 'Publications' and year==2013").counts.iloc[0]
n_2023 = counts_normalised_df.query("Dataset == 'Publications' and year==2023").counts.iloc[0]
growth = (n_2023 - n_2013) / n_2013
print(f"Number of publications in 2013: {n_2013}")
print(f"Number of publications in 2023: {n_2023}")
print(f"Growth in publications: {growth:.2f}")

In [ ]:
savename = 'documents_by_year'
AltairSaver.save(fig, savename)

In [ ]:
fig_title = "Percentage of documents by year"
fig_subtitle = ["Normalised with respect to the total number of publications/patents"]

fig = pu.ts_smooth(
    (
        counts_normalised_df
        .query('year < 2024')
    ),
    ["Publications", "Patents"],
    variable= "percentage",
    variable_title = "",
    category_column = "Dataset",
    width = 300,
    height = 150,
    legend_orient='right',
)
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle)
fig

In [ ]:
savename = 'percentage_documents_by_year'
AltairSaver.save(fig, savename)

## Detection vs management

In [ ]:
n_management = relevant_df.Management_.sum()
n_detection = relevant_df.Detection_.sum()
n = len(relevant_df)

print(f"Number of documents about detection: {n_detection}")
print(f"Number of documents about management: {n_management}")

print(f"Fraction of documents about detection: {n_detection/n:.2f}")
print(f"Fraction of documents about management: {n_management/n:.2f}")

In [ ]:
n_both = len(relevant_df.query("Detection_ == 1 and Management_ == 1"))
print(f"Number of documents about both detection and management: {n_both}")
print(f"Fraction of documents about both detection and management: {n_both/n:.2f}")

In [ ]:
# Number of documents per Dataset that are about detection or management
detection_management_df = (
    relevant_df
    .groupby(['Dataset'])
    .agg(
        n=('id', 'count'),
        Detection=('Detection_', 'sum'),
        Management=('Management_', 'sum'),
    )
    .assign(
        fraction_detection = lambda df: df['Detection'] / df['n'],
        fraction_management = lambda df: df['Management'] / df['n'],
    )
    .reset_index()
)
detection_management_df

In [ ]:
detection_management_df

In [ ]:
# a bar chart with the number of documents about detection and management for each dataset
fig_title = "Number of documents about detection or management"
fig_df = (
    detection_management_df
    # make it a long dataframe
    .melt(id_vars=['Dataset', 'n'], var_name='Type', value_name='Count')
    .query('Type in ["Detection", "Management"]')    
)

fig = (
    alt.Chart(fig_df)
    .mark_bar()
    .encode(
        x=alt.X('Count:Q', title=''),
        y=alt.Y('Type:N', title='', sort='-x'),
        color=alt.Color('Dataset:N', title=''),
        tooltip=['Dataset', 'Count', 'Type']
    )
)
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, "")
fig

In [ ]:
savename = 'detection_management_documents'
AltairSaver.save(fig, savename)

## Topics

In [ ]:
topics_counts = (
    exploded_relevant_df
    .groupby('topics_list')
    .agg(counts=('id', 'count'))
    .reset_index()
    .merge(topics_df[['topic', 'Type', 'name']], left_on='topics_list', right_on='topic', how='left')
    .drop(columns=['topics_list'])
    .sort_values(['Type', 'counts'], ascending=[True, False])
)

topics_pubs_counts = (
    exploded_relevant_df
    .query('Dataset == "Publications"')
    .groupby('topics_list')
    .agg(counts=('id', 'count'))
    .reset_index()
    .merge(topics_df[['topic', 'Type', 'name']], left_on='topics_list', right_on='topic', how='left')
    .drop(columns=['topics_list'])
)

topics_patent_counts = (
    exploded_relevant_df
    .query('Dataset == "Patents"')
    .groupby('topics_list')
    .agg(counts=('id', 'count'))
    .reset_index()
    .merge(topics_df[['topic', 'Type', 'name']], left_on='topics_list', right_on='topic', how='left')
    .drop(columns=['topics_list'])
)

topics_counts_datasets = (
    pd.concat([
        topics_pubs_counts.assign(Dataset='Publications'),
        topics_patent_counts.assign(Dataset='Patents'),
    ], ignore_index=True)
)
# topics_counts = (
#     topics_counts
#     .merge(topics_pubs_counts, on=['topic', 'Type', 'name'], how='left', suffixes=('', '_pubs'))
#     .merge(topics_patent_counts, on=['topic', 'Type', 'name'], how='left', suffixes=('', '_patents'))
#     .fillna(0)
#     .sort_values(['Type', 'counts'], ascending=[True, False])
# )

In [ ]:
# Stacked bar chart
fig_title = "Number of documents by topic"
fig_subtitle = ""

sort_order = topics_counts.name.to_list()

fig = (
    alt.Chart(topics_counts_datasets, height=500)
    .mark_bar()
    .encode(
        x=alt.X('counts:Q', title=''),
        # sort according to the dataframe sort order
        y=alt.Y('name:N', title='', sort=sort_order),
        color=alt.Color('Dataset:N', title=''),
        tooltip=['name', 'counts', 'Dataset']
    )
)

# axis on top as well
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle)
fig

In [ ]:
savename = 'number_of_documents_by_topic'
AltairSaver.save(fig, savename)

## Overall stats

In [ ]:
df_patent_vs_openalex = (
    exploded_relevant_df
    .query("year >= 2013 and year < 2024")
    .groupby(['topics_list', 'source'])
    .agg(counts=('id', 'count'))
    .reset_index()
    .merge(topics_df[['topic', 'type', 'name']], left_on='topics_list', right_on='topic', how='left')
    .drop(columns='topics_list')
)

In [ ]:
n_patents = len(
    relevant_df
    .query('year >= 2013 and year < 2024')
    .query('source == "patents"')
)

n_openalex = len(
    relevant_df
    .query('year >= 2013 and year < 2024')
    .query('source == "openalex"')
)

In [ ]:
# pivot the table so that values in source column become columns and counts are the values
df_patent_vs_openalex_pivot = (
    df_patent_vs_openalex
    .pivot_table(index=['type', 'name'], columns='source', values='counts', fill_value=0)
    .reset_index()
    .assign(
        patents_normalised=lambda df: df['patents'] / n_patents,
        openalex_normalised=lambda df: df['openalex'] / n_openalex
    )
    .assign(
        difference=lambda df: df['openalex_normalised'] - df['patents_normalised']
    )
)
df_patent_vs_openalex_pivot


In [ ]:
# plot a scatterplot with difference on x-axis and topic name on y-axis and sort by y-axis values
alt.Chart(
    df_patent_vs_openalex_pivot,
    width=500
).mark_circle(size=100).encode(
    x='difference:Q',
    y=alt.Y('name:N', sort='-x'),
    color='type:N',
    # size='openalex_normalised:Q'
)

## Co-occurrences

In [ ]:
coocc_df = (
    exploded_relevant_df
    .groupby("id")
    .agg(topic_list = ("topics_list", list))
    .reset_index()
)

# function to count all pairs of topics, use the topics_df unique topics, and coocc_df for topics in each document
# return a dataframe with index and columns as topics and values as counts
def get_cooccurrences(topics_df, coocc_df):
    cooccurrences = pd.DataFrame(index=topics_df['topic'], columns=topics_df['topic']).fillna(0)
    for topic_list in coocc_df['topic_list']:
        for i, topic1 in enumerate(topic_list):
            for j, topic2 in enumerate(topic_list):
                if i != j:
                    cooccurrences.loc[topic1, topic2] += 1
    return cooccurrences

matrix = get_cooccurrences(topics_df, coocc_df)

In [ ]:
matrix.index.name = 'topic1'

In [ ]:
# turn matrix into long format
matrix_long = (
    matrix
    .stack()
    .reset_index()
    .rename(columns={0: 'counts', 'topic': 'topic2'})
    .merge(topics_df[['topic', 'name']], left_on='topic1', right_on='topic').rename(columns={'name': 'name1'}).drop(columns='topic')
    .merge(topics_df[['topic', 'name']], left_on='topic2', right_on='topic').rename(columns={'name': 'name2'}).drop(columns='topic')    
)
matrix_long

In [ ]:
(
    matrix_long
    .to_csv("cooccurrences.csv", index=False)
)

In [ ]:
# visualise the matrix using a heatmap
# sort using a specified order
sort_order = topics_df.topic.to_list()
alt.Chart(
    matrix_long,
    width=500,
    height=500,
).mark_rect().encode(
    x=alt.X('name1:N', sort=sort_order),
    y=alt.Y('name2:N', sort=sort_order),
    color='counts:Q',
    tooltip=['name1', 'name2', 'counts']
)

In [ ]:
# normalise each row by the sum of the row
matrix_normalised = matrix.div(matrix.sum(axis=1), axis=0)
# turn matrix into long format
matrix_long_normalised = (
    matrix_normalised
    .stack()
    .reset_index()
    .rename(columns={0: 'counts', 'topic': 'topic2'})
    .merge(topics_df[['topic', 'name']], left_on='topic1', right_on='topic').rename(columns={'name': 'name1'}).drop(columns='topic')
    .merge(topics_df[['topic', 'name']], left_on='topic2', right_on='topic').rename(columns={'name': 'name2'}).drop(columns='topic')     
)

In [ ]:
# visualise the matrix using a heatmap
# sort using a specified order
sort_order = topics_df.topic.to_list()
alt.Chart(
    matrix_long_normalised,
    width=500,
    height=500,
).mark_rect().encode(
    x=alt.X('name1:N', sort=sort_order),
    y=alt.Y('name2:N', sort=sort_order),
    color='counts:Q',
    tooltip=['topic1', 'topic2', 'counts']
)

## Co-occurrences of specific topics

In [ ]:
matrix_long.head()

In [ ]:
types = ['Area of learning', 'Health', 'Development', 'Social', 'General']
topic = "ai2"
df = (
    matrix_long
    .query("topic1 == 'ai2'")
    .merge(topics_df[['topic', 'Type']], left_on='topic2', right_on='topic')
    .drop(columns='topic')
    .rename(columns={'Type': 'Type2'})
    .query("Type2 in @types")
)
df

In [ ]:
fig = alt.Chart(
    df,
    width=300,
    height=400,
).mark_bar().encode(
    y=alt.Y('name2:N', sort='-x', title=''),
    x=alt.X('counts:Q', title=''),
    color=alt.Color('Type2:N', legend=alt.Legend(title='Type')),
    tooltip=['name2', 'counts']
)

fig = pu.configure_titles(pu.configure_plots(fig), "Applications of AI", "Number of documents tagged with AI and other topics")
fig

In [ ]:
savename = 'applications_of_ai'
AltairSaver.save(fig, savename)

In [ ]:
topic = "mobile"
df = (
    matrix_long
    .query("topic1 == @topic")
    .merge(topics_df[['topic', 'Type']], left_on='topic2', right_on='topic')
    .drop(columns='topic')
    .rename(columns={'Type': 'Type2'})
    .query("Type2 in @types")
)
fig = alt.Chart(
    df,
    width=300,
    height=400,
).mark_bar().encode(
    y=alt.Y('name2:N', sort='-x', title=''),
    x=alt.X('counts:Q', title=''),
    color=alt.Color('Type2:N', legend=alt.Legend(title='Type')),
    tooltip=['name2', 'counts']
)

fig = pu.configure_titles(pu.configure_plots(fig), "Applications of AI", "Number of documents tagged with AI and other topics")
fig

In [ ]:
savename = 'applications_of_mobile'
AltairSaver.save(fig, savename)

In [ ]:
types = ['Area of learning', 'Health', 'Development', 'Social', 'General', 'Innovation']
topic = "parenting2"
df = (
    matrix_long
    .query("topic1 == @topic")
    .merge(topics_df[['topic', 'Type']], left_on='topic2', right_on='topic')
    .drop(columns='topic')
    .rename(columns={'Type': 'Type2'})
    .query("Type2 in @types")
)
fig = alt.Chart(
    df,
    width=300,
    height=500,
).mark_bar().encode(
    y=alt.Y('name2:N', sort='-x', title=''),
    x=alt.X('counts:Q', title=''),
    color=alt.Color('Type2:N', legend=alt.Legend(title='Type')),
    tooltip=['name2', 'counts']
)

fig = pu.configure_titles(pu.configure_plots(fig), "Overlaps with Parenting category", "Number of documents tagged with Parenting and other topics")
fig

In [ ]:
savename = 'applications_of_parenting'
AltairSaver.save(fig, savename)

In [ ]:
types = ['Area of learning', 'Health', 'Development', 'Social', 'General', 'Innovation']
topic = "income"
df = (
    matrix_long
    .query("topic1 == @topic")
    .merge(topics_df[['topic', 'Type']], left_on='topic2', right_on='topic')
    .drop(columns='topic')
    .rename(columns={'Type': 'Type2'})
    .query("Type2 in @types")
)
fig = alt.Chart(
    df,
    width=300,
    height=500,
).mark_bar().encode(
    y=alt.Y('name2:N', sort='-x', title=''),
    x=alt.X('counts:Q', title=''),
    color=alt.Color('Type2:N', legend=alt.Legend(title='Type')),
    tooltip=['name2', 'counts']
)

fig = pu.configure_titles(pu.configure_plots(fig), "Overlaps with Income", "Number of documents tagged with Income and other topics")
fig

In [ ]:
savename = 'applications_of_income'
AltairSaver.save(fig, savename)

## Co-occurences over time

In [ ]:
coocc_time_df = (
    exploded_relevant_df
    .groupby(["id", "year"])
    .agg(topic_list = ("topics_list", list))
    .reset_index()
)

In [ ]:
def get_cooccurrences_growth(topic1):
    topics2 = topics_df.topic.to_list()
    start_period = [2014, 2018]
    end_period = [2019, 2023]

    overlap_growth = []
    for topic in topics2:
        df = (
            coocc_time_df
            .assign(
                cooccurs = lambda df: df['topic_list'].apply(lambda x: topic1 in x and topic in x)
            )
            .groupby('year')
            .agg(counts=('cooccurs', 'sum'))
            .reset_index()
        )
        # sum the counts for the two periods
        sum_start_period = df.query("year >= @start_period[0] and year <= @start_period[1]").counts.sum()
        sum_end_period = df.query("year >= @end_period[0] and year <= @end_period[1]").counts.sum()
        growth_period  = (sum_end_period - sum_start_period) / sum_start_period
        overlap_growth.append({
            'topic': topic,
            'sum_start_period': sum_start_period,
            'sum_end_period': sum_end_period,
            'growth': growth_period
        })
    return pd.DataFrame(overlap_growth)

In [ ]:
topics_of_interest = ["preschool", "literacy", "mathematics", "communication"]
cooc_growth_df = (
    get_cooccurrences_growth("ai2")
    .sort_values('growth', ascending=False)
)
df_ai2 = cooc_growth_df.query("topic in @topics_of_interest")

In [ ]:
topics_of_interest = ["preschool", "literacy", "mathematics", "communication"]
cooc_growth_df = (
    get_cooccurrences_growth("mobile")
    .sort_values('growth', ascending=False)
)
df_mobile = cooc_growth_df.query("topic in @topics_of_interest")

In [ ]:
coo_growth_df_ = (
    pd.concat([df_ai2.assign(topic1="AI"), df_mobile.assign(topic1="Mobile")], ignore_index=True)
    .assign(
        category = lambda df: df.topic1 + ' + ' + df.topic,
        magnitude = lambda df: df.sum_start_period + df.sum_end_period
    )
)
coo_growth_df_

In [ ]:
fig_title = "AI, Mobile and areas of learning"
fig_subtitle = "Growth and total number of documents, 2014-2023"

fig = (
    alt.Chart(
        coo_growth_df_,
    )
    .mark_point()
    .encode(
        # add title
        x=alt.X('magnitude:Q', axis=alt.Axis(title='Number of documents')),
        # add percentages to the growth
        y=alt.Y('growth:Q', axis=alt.Axis(format='%'), title='Growth'),
        text='category:N',
        # color='Type',
        # tooltip=['category', 'magnitude', alt.Tooltip('growth', format='.2%')]
    )
    .properties(
        width=350,
        height=350  
    )
)

labels = fig.mark_text(
    align='left',
    baseline='middle',
    dx=7
).encode(
    text='category:N'
)

# Add a dashed line for zero growth
zero_growth = alt.Chart(pd.DataFrame({'zero': [0]})).mark_rule(color='black', strokeDash=[3,3]).encode(y='zero')

fig = (
    (fig + labels + zero_growth)
)
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle).configure_axis(grid=False)
fig

In [ ]:
savename = "ai_mobile_areas_of_learning"
# AltairSaver.save(fig, savename)

In [ ]:
topic1 = "mobile"
topic = "parenting2"
df = (
    coocc_time_df
    .assign(
        cooccurs = lambda df: df['topic_list'].apply(lambda x: topic1 in x and topic in x)
    )
    .query("cooccurs")
    .merge(relevant_df[['id', 'Dataset', 'text']], how='left', on='id')
    .sort_values('year', ascending=False)
)

In [ ]:
pd.set_option('display.max_colwidth', None)
# df.query("Dataset == 'Patents'")
# df


In [ ]:
(
    get_cooccurrences_growth("parenting2")
    .sort_values('growth', ascending=False)
)

In [ ]:
(
    get_cooccurrences_growth("income")
    .sort_values('growth', ascending=False)
)

## Magnitude and growth

In [ ]:
# # Total baseline
# total_baseline_df = (
#     baseline_df
#     .groupby('year')
#     .agg(total_counts=('total_counts', 'sum'))
#     .reset_index()
# )

In [ ]:
def get_timeseries(df: pd.DataFrame, normalise: bool=False, baseline_df: pd.DataFrame=None) -> pd.DataFrame:
    df = (
        df
        .groupby(['year'])
        .agg(counts=('id', 'count'))
        .reset_index()
    )
    # add 0 for empty years
    years = list(range(2013, 2024))
    df = (
        df
        .merge(
            pd.DataFrame({'year': years}),
            on='year',
            how='right'
        )
        .fillna(0)
    )
    # normalise by baseline
    if normalise:
        df = (
            df
            .merge(
                baseline_df,
                on='year',
                how='left'
            )
            .assign(
                fraction = lambda df: df['counts'] / df['total_counts'],
                percentage = lambda df: df['fraction'] * 100
            )
        )
    return df.sort_values('year')


def get_timeseries_charts(df: pd.DataFrame, topics_dict: dict, baseline_df: pd.DataFrame=None) -> pd.DataFrame:
    timeseries = []
    for topic in topics_dict:
        t_df = get_timeseries(
            df.query("topics_list == @topic"), 
            normalise=True, 
            baseline_df=baseline_df
        ).assign(category=topic)
        timeseries.append(t_df)
    return pd.concat(timeseries, ignore_index=True)


# Calculate magnitude and growth for each category
# Magnitude is the average across past five years (2019-2023)
# Growth is the average growth rate across past five years (2019-2023)
# Growth is calculated by taking 3-year moving average and then comparing 2019 and 2023
def calculate_magnitude_growth(timeseries_df: pd.DataFrame) -> pd.DataFrame:
    magnitude_growth = []
    for category in timeseries_df['category'].unique():
        category_df = timeseries_df.query('category == @category').sort_values('year')
        magnitude = category_df.query('year >= 2019 and year <= 2023')['counts'].sum()
        magnitude_normalised = category_df.query('year >= 2019 and year <= 2023')['fraction'].mean()
        # calculate moving average with sliding window
        category_df['moving_average'] = category_df['fraction'].rolling(window=3).mean()
        growth = (category_df.query('year == 2023')['moving_average'].values[0] - category_df.query('year == 2019')['moving_average'].values[0]) / category_df.query('year == 2019')['moving_average'].values[0]
        magnitude_growth.append({
            'category': category,
            'magnitude': magnitude,
            'magnitude_normalised': magnitude_normalised,
            'growth': growth
        })
    return pd.DataFrame(magnitude_growth)

# def timeseries_chart(df: pd.DataFrame, title: str) -> alt.Chart:
#     return (
#         alt.Chart(
#             df,
#             width=500)
#         .mark_line()
#         .encode(
#             x='year:O',
#             y='counts:Q'
#         )
#         .properties(
#             title=title
#         )
#     )


In [ ]:
au.ts_magnitude_growth_(
    ts_df = (
        counts_normalised_df
        .query("Dataset == 'Publications' and year < 2024")
    ),
    year_start = 2019,
    year_end = 2023   
)

In [ ]:
au.ts_magnitude_growth_(
    ts_df = (
        counts_normalised_df
        .query("Dataset == 'Patents' and year < 2024")
    ),
    year_start = 2019,
    year_end = 2023   
)

### Publications

In [ ]:
pubs_df = exploded_relevant_df.query("Dataset == 'Publications'")
pubs_timeseries_df = get_timeseries_charts(pubs_df, topics_dict, openalex_baseline)
pubs_magnitude_growth_df = (
    calculate_magnitude_growth(pubs_timeseries_df)
    .merge(
        topics_df,
        left_on='category',
        right_on='topic',
        how='left'
    )
    .drop(columns=['topic'])
)

In [ ]:
# category_df = pubs_timeseries_df.query('category == "infancy"').sort_values('year')
# au.ts_magnitude_growth_(category_df, 2019, 2023)


In [ ]:
pubs_magnitude_growth_df.head()

In [ ]:
# # Plot the technology timeseries with AI, Mobile, AR/VR, Social media
# # include = ['ai2', 'mobile', 'ar_vr', 'social_media']
# include = topics_df.query('type == "Development"').topic.to_list()
# fig = (
#     alt.Chart(
#         timeseries_df.query('category in @include').query('year < 2024'),
#         width=500)
#     .mark_line()
#     .encode(
#         x='year:O',
#         y='counts:Q',
#         color='category',
#         tooltip='category'
#     )
#     .properties(
#         title='Document counts (patents and publications)'
#     )
# )
# fig.interactive()

In [ ]:
# Plot a scatter plot of magnitude and growth
# Display labels next to the scatter points and remove grid lines
fig_title = "Publication magnitude and growth by topic"
fig_subtitle = "Growth and total number of publications, 2019-2023"

fig = (
    alt.Chart(
        pubs_magnitude_growth_df,
    )
    .mark_point()
    .encode(
        # add title
        x=alt.X('magnitude:Q', axis=alt.Axis(title='Number of documents')),
        # add percentages to the growth
        y=alt.Y('growth:Q', axis=alt.Axis(format='%'), title='Growth'),
        text='category:N',
        color='Type',
        tooltip=['category', 'magnitude', alt.Tooltip('growth', format='.2%')]
    )
    .properties(
        width=400,
        height=400  
    )
)

labels = fig.mark_text(
    align='left',
    baseline='middle',
    dx=7
).encode(
    text='name:N'
)

# Add a dashed line for zero growth
zero_growth = alt.Chart(pd.DataFrame({'zero': [0]})).mark_rule(color='black', strokeDash=[3,3]).encode(y='zero')

fig = (
    (fig + labels + zero_growth)
)
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle).configure_axis(grid=False)

In [ ]:
fig

In [ ]:
savename = 'publication_magnitude_growth'
AltairSaver.save(fig, savename)

In [ ]:
# Plot x-axis as growth, size as magnitude, and y axis as topic
fig_title = "Publication magnitude and growth by topic"
fig_subtitle = "Growth and total number of publications, 2019-2023"

def growth_figure(pubs_magnitude_growth_df):
    fig = (
        alt.Chart(
            pubs_magnitude_growth_df,
        )
        .mark_circle()
        .encode(
            # add title
            x=alt.X('growth:Q', axis=alt.Axis(format='%', labelAlign='center'), title='Growth'),
            # don't show the axis
            y=alt.Y('name:N', title='Topic', sort='-x', axis=None),
            size=alt.Size('magnitude:Q', title='Number of documents', legend=None),
            color=alt.Color('Type', legend=alt.Legend(title='', orient='bottom-right')),
            tooltip=['name', 'magnitude', alt.Tooltip('growth', format='.2%')]
        )
        .properties(
            width=400,
            height=400  
        )
    )

    # Labels but same size for all
    fig_labels_ = (
        alt.Chart(
            pubs_magnitude_growth_df,
        )
        .mark_circle()
        .encode(
            # add title
            x=alt.X('growth:Q', axis=alt.Axis(format='%'), title='Growth'),
            y=alt.Y('name:N', title='Topic', sort='-x'),
        )
    )
    fig_labels = (
        fig_labels_.mark_text(
            align='left',
            baseline='middle',
            dx=7
        ).encode(
            text='name:N'
        )
    )

    # Add a dashed vertical line for zero growth
    zero_growth = alt.Chart(pd.DataFrame({'zero': [0]})).mark_rule(color='black', strokeDash=[3,3]).encode(x='zero')

    fig = fig + fig_labels + zero_growth
    return fig

fig = growth_figure(pubs_magnitude_growth_df)
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle)#.configure_axis(grid=False)
fig

In [ ]:
savename = 'publication_growth'
AltairSaver.save(fig, savename)

### Time series

In [ ]:
fig = pu.ts_smooth(
    pubs_timeseries_df,
    ["internet", "health", "mental_health", "genetics", "rct", "nutrition"],
    variable= "percentage",
    variable_title = "",
    category_column = "category",
    width = 400,
    height = 200,
    legend_orient='right',
)
fig = pu.configure_titles(pu.configure_plots(fig), "Percentage of documents by year", "Normalised with respect to the total number of publications")
fig

### Patents

In [ ]:
patents_df = (
    exploded_relevant_df
    .query("Dataset == 'Patents'")
    # .query("country_code != 'CN'")
)
patents_timeseries_df = get_timeseries_charts(patents_df, topics_dict, patents_baseline)
patents_magnitude_growth_df = (
    calculate_magnitude_growth(patents_timeseries_df)
    .merge(
        topics_df,
        left_on='category',
        right_on='topic',
        how='left'
    )
    .drop(columns=['topic'])
    .query('magnitude >= 25')
)

In [ ]:
# Plot a scatter plot of magnitude and growth
# Display labels next to the scatter points and remove grid lines
fig_title = "Patent magnitude and growth by topic"
fig_subtitle = "Growth and total number of patents, 2019-2023"

fig = (
    alt.Chart(
        patents_magnitude_growth_df,
    )
    .mark_point()
    .encode(
        # add title
        x=alt.X('magnitude:Q', axis=alt.Axis(title='Number of documents')),
        # add percentages to the growth
        y=alt.Y('growth:Q', axis=alt.Axis(format='%'), title='Growth'),
        text='category:N',
        color='Type',
        tooltip=['category', 'magnitude', alt.Tooltip('growth', format='.2%')]
    )
    .properties(
        width=400,
        height=400  
    )
)

labels = fig.mark_text(
    align='left',
    baseline='middle',
    dx=7
).encode(
    text='name:N'
)

# Add a dashed line for zero growth
zero_growth = alt.Chart(pd.DataFrame({'zero': [0]})).mark_rule(color='black', strokeDash=[3,3]).encode(y='zero')

fig = (
    (fig + labels + zero_growth)
)
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle).configure_axis(grid=False)

In [ ]:
fig

In [ ]:
savename = 'patent_magnitude_growth'
AltairSaver.save(fig, savename)

In [ ]:
baseline_growth = alt.Chart(pd.DataFrame({'zero': [-0.17731658]})).mark_rule(color='black', strokeDash=[3,3]).encode(x='zero')
fig = growth_figure(patents_magnitude_growth_df) + baseline_growth
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle)#.configure_axis(grid=False)
fig

In [ ]:
savename = 'patent_growth'
AltairSaver.save(fig, savename)

### Patents (without China)

In [ ]:
patents_df = (
    exploded_relevant_df
    .query("Dataset == 'Patents'")
    .query("country_code != 'CN'")
)
patents_timeseries_df = get_timeseries_charts(patents_df, topics_dict, patents_baseline)
patents_magnitude_growth_df = (
    calculate_magnitude_growth(patents_timeseries_df)
    .merge(
        topics_df,
        left_on='category',
        right_on='topic',
        how='left'
    )
    .drop(columns=['topic'])
    .query('magnitude >= 25')
)

In [ ]:
baseline_growth = alt.Chart(pd.DataFrame({'zero': [-0.17731658]})).mark_rule(color='black', strokeDash=[3,3]).encode(x='zero')
fig = growth_figure(patents_magnitude_growth_df) + baseline_growth
fig = pu.configure_titles(pu.configure_plots(fig), fig_title, fig_subtitle)#.configure_axis(grid=False)
fig

## Ad-hoc checks

In [ ]:
# Coronavirus mentions
keywords = ["covid", "coronavirus"]
hits_df = (
    relevant_df[['id', 'year', 'text']]
    .copy()
    .assign(hits = lambda df: df.text.str.lower().str.contains("|".join(keywords)))
    .query('hits == True')
)
hits_counts_df = hits_df.groupby('year').agg(counts=('id', 'count')).reset_index().assign(cat="cat")

pu.ts_smooth(
    hits_counts_df.query('year < 2024'),
    ["cat"],
    variable= "counts",
    variable_title = "",
    category_column = "cat",
    width = 300,
    height = 150,
    legend_orient='right',
)

In [ ]:
hits_df

## Mock use cases: AI

In [ ]:
# Calculate embeddings, reduce dimensionality, cluster and plot the embeddings
import discovery_child_development.utils.cluster_analysis_utils as cau
from discovery_child_development import logger
alt.data_transformers.disable_max_rows()
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

SEED = 42
UMAP_PARAMS = {
    "n_components": 50,
    "n_neighbors": 10,
    "min_dist": 0.5,
    "spread": 0.5,
}

In [ ]:
data_df = get_hits(relevant_df, keywords['AI'])

In [ ]:
# Create embeddings for the unique concepts
embeddings = model.encode(data_df["text"].tolist(), show_progress_bar=True)

In [ ]:
# Reduce dimensionality of the embeddings
embeddings_50 = cau.umap_reducer(embeddings, UMAP_PARAMS, random_umap_state=SEED)

# Run with an arbitrary number of clusters
kmeans_labels = cau.kmeans_clustering(
    embeddings_50,
    kmeans_params={"init": "k-means++", "n_clusters": 6, "random_state": SEED},
)

# Reduce original vectors to 2D for plotting
embeddings_2d = cau.reduce_to_2D(embeddings, random_state=SEED)


In [ ]:
# Add 2D vectors into the dataframe for plotting
clusters_df = (
    data_df.copy()
    .reset_index(drop=True)
    .assign(
        cluster=kmeans_labels,
        x=embeddings_2d[:, 0],
        y=embeddings_2d[:, 1],
    )
)

In [ ]:
import importlib
importlib.reload(cau)

In [ ]:
CLUSTER_SUMMARY_MESSAGE = "Here are the most central documents of cluster. \
Describe what kind of innovations is this cluster capturing, in 2 sentences. \
\n\n##Abstracts\n\n {} \n\n##Description (2 short sentences)"

cluster_descriptions = cau.describe_clusters_with_gpt(
    cluster_df=clusters_df,
    embeddings=embeddings,
    n_central=30,
    gpt_message=CLUSTER_SUMMARY_MESSAGE,
)
cluster_names_dict = cau.generate_cluster_names_with_gpt(
    cluster_descriptions=cluster_descriptions,
)
cluster_summaries = pd.DataFrame(
    data={
        "cluster": cluster_names_dict.keys(),
        "cluster_name": cluster_names_dict.values(),
        "cluster_description": cluster_descriptions,
    }
)
pd.set_option("display.max_colwidth", None)
cluster_summaries

In [ ]:
clusters_df_final = (
    clusters_df.copy().merge(
    cluster_summaries, left_on="cluster", right_on="cluster", how="left")
    .assign(url = lambda df: [prepare_url(row.id, row.source) for i, row in df.iterrows()])
)

fig = (
    alt.Chart(clusters_df_final[['x', 'y', "source", "cluster", "cluster_name", "cluster_description", "id", "text", "url"]])
    .mark_circle()
    .encode(
        x="x",
        y="y",
        color=alt.Color("cluster_name:N", legend=alt.Legend(title="cluster name")),
        tooltip=["cluster", "cluster_name","text"],
        # change symbol type depending on source
        shape="source",
        # add url
        href = 'url:N'
    )
    .properties(width=800, height=600)
    .interactive()
)

# create labels for cluster centroids
# calculate the centroid of each cluster
# add the labels to the plot
centroids_df = (
    clusters_df_final
    .groupby('cluster')
    .agg(x=('x', 'mean'), y=('y', 'mean'))
    .reset_index()
)
# add names
centroids_df = centroids_df.merge(cluster_summaries[['cluster', 'cluster_name']], on='cluster', how='left')

labels = (
    alt.Chart(centroids_df)
    .mark_text(align='left', baseline='middle', dx=7)
    .encode(
        x='x:Q',
        y='y:Q',
        # increase font size
        text=alt.Text('cluster_name:N')
    )
)

(
    (fig + labels)
    .configure_axis(grid=False)
)

In [ ]:
fig.save(PROJECT_DIR / 'outputs/enrichments/AI_test_dataset.html')

In [ ]:
clusters_df_final.to_csv(PROJECT_DIR / 'outputs/enrichments/AI_test_dataset.csv', index=False)

## Check categorised data

In [ ]:
topics = [
    "ai2",
    "ar_vr",
    "mobile",
    "income",
    "parenting",
]

In [ ]:
LABELS_DIR = PROJECT_DIR / 'outputs/labels/taxonomy_cat'
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"
topics = json.load(open(PATH_TO_TOPICS, 'r'))

In [ ]:
detection_df = pd.read_csv(DATA_DIR / 'openalex_patents_detection_labels.csv')

In [ ]:
ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'
labelled_df = pd.read_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_all.csv')

In [ ]:
relevant_labelled_df = (
    relevant_df
    .assign(
        id = lambda df: df['id'].apply(lambda x: x.split('/')[-1])
    )
    .merge(labelled_df, on='id', how='left')
    .assign(topic_name = lambda df: df['topic'].apply(lambda x: topics[x]['name'] if pd.notnull(x) else None))
    # .drop("prediction", axis=1)
)

In [ ]:
relevant_labelled_df.groupby(["source", "topic_name"]).size()

In [ ]:
(
    relevant_labelled_df
    .query("topic == 'ai2'")
    .prediction

In [ ]:
def get_timeseries_charts(df: pd.DataFrame, keywords: dict) -> alt.Chart:
    keyword_timeseries = []
    for keyword, keywords_list in keywords.items():
        keyword_hits_df = get_hits(df, keywords_list)
        keyword_timeseries.append(get_timeseries(keyword_hits_df).assign(category=keyword))
    return pd.concat(keyword_timeseries, ignore_index=True)

timeseries_df = get_timeseries_charts(relevant_df, keywords)

## Check each category and output a cluster map

In [ ]:
# Calculate embeddings, reduce dimensionality, cluster and plot the embeddings
import discovery_child_development.utils.cluster_analysis_utils as cau
from discovery_child_development import logger
alt.data_transformers.disable_max_rows()
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

SEED = 42
UMAP_PARAMS = {
    "n_components": 50,
    "n_neighbors": 10,
    "min_dist": 0.5,
    "spread": 0.5,
}

In [ ]:
VECTORS_PATH = "data/outputs/vectors/"
VECTORS_FILE = "sentence_vectors_384_labelled.parquet"
from discovery_child_development.getters.openalex import get_sentence_embeddings
from discovery_child_development import S3_BUCKET

# Embeddings from all-MiniLM-L6-v2
embeddings_all = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET, filepath=VECTORS_PATH, filename=VECTORS_FILE, id="id",
    )
    .reset_index()
    .assign(id=lambda df: df['id'].apply(lambda x: x.split('/')[-1]))
    # restore index
    # .set_index("id")
)

In [ ]:
topic = "ai2"
topic_df = (
    relevant_labelled_df.query('topic == @topic')
    .merge(
        embeddings_all,
        on='id',
        how='left'
    )
)
embeddings = topic_df.miniLM_384_vector.values.tolist()

In [ ]:
# Reduce dimensionality of the embeddings
embeddings_50 = cau.umap_reducer(embeddings, UMAP_PARAMS, random_umap_state=SEED)

# Run with an arbitrary number of clusters
kmeans_labels = cau.kmeans_clustering(
    embeddings_50,
    kmeans_params={"init": "k-means++", "n_clusters": 6, "random_state": SEED},
)

# Reduce original vectors to 2D for plotting
embeddings_2d = cau.reduce_to_2D(embeddings, random_state=SEED)


In [ ]:
# Add 2D vectors into the dataframe for plotting
clusters_df = (
    topic_df.copy()
    .reset_index(drop=True)
    .assign(
        cluster=kmeans_labels,
        x=embeddings_2d[:, 0],
        y=embeddings_2d[:, 1],
    )
)

In [ ]:
import numpy as np

CLUSTER_SUMMARY_MESSAGE = "Here are the most central documents of cluster. \
Describe what kind of innovations is this cluster capturing, in 2 sentences. \
\n\n##Abstracts\n\n {} \n\n##Description (2 short sentences)"

cluster_descriptions = cau.describe_clusters_with_gpt(
    cluster_df=clusters_df,
    embeddings=np.array(embeddings),
    n_central=30,
    gpt_message=CLUSTER_SUMMARY_MESSAGE,
)
cluster_names_dict = cau.generate_cluster_names_with_gpt(
    cluster_descriptions=cluster_descriptions,
)
cluster_summaries = pd.DataFrame(
    data={
        "cluster": cluster_names_dict.keys(),
        "cluster_name": cluster_names_dict.values(),
        "cluster_description": cluster_descriptions,
    }
)
pd.set_option("display.max_colwidth", None)
cluster_summaries

In [ ]:
clusters_df_final = (
    clusters_df.copy().merge(
    cluster_summaries, left_on="cluster", right_on="cluster", how="left")
    .assign(url = lambda df: [prepare_url(row.id, row.source) for i, row in df.iterrows()])
)

fig = (
    alt.Chart(clusters_df_final[['x', 'y', "source", "cluster", "cluster_name", "cluster_description", "id", "text", "url"]])
    .mark_circle()
    .encode(
        x="x",
        y="y",
        color=alt.Color("cluster_name:N", legend=alt.Legend(title="cluster name")),
        tooltip=["cluster", "cluster_name","text"],
        # change symbol type depending on source
        shape="source",
        # add url
        href = 'url:N'
    )
    .properties(width=800, height=600)
    .interactive()
)

# create labels for cluster centroids
# calculate the centroid of each cluster
# add the labels to the plot
centroids_df = (
    clusters_df_final
    .groupby('cluster')
    .agg(x=('x', 'mean'), y=('y', 'mean'))
    .reset_index()
)
# add names
centroids_df = centroids_df.merge(cluster_summaries[['cluster', 'cluster_name']], on='cluster', how='left')

labels = (
    alt.Chart(centroids_df)
    .mark_text(align='left', baseline='middle', dx=7)
    .encode(
        x='x:Q',
        y='y:Q',
        # increase font size
        text=alt.Text('cluster_name:N')
    )
)

(
    (fig + labels)
    .configure_axis(grid=False)
)

## Prepare data for exploration

In [28]:
detection_df = pd.read_csv(ENRICHED_DATA_DIR / 'openalex_patents_detection_labels.csv')

In [29]:
from discovery_child_development.utils.utils import prepare_url

In [30]:
relevant_labelled_df.head(1)

NameError: name 'relevant_labelled_df' is not defined

In [ ]:
relevant_labelled_df.query("topic == 'ai2'").query("prob_relevant >= 0.5").groupby("source").size()

In [ ]:
titles_df = (
    pd.concat([
        patents_metadata_df[['publication_number', 'title']].rename(columns={'publication_number': 'id'}),
        (
            openalex_metadata_df
            .rename(columns={'openalex_id': 'id'})
            .assign(id = lambda df: df['id'].apply(lambda x: x.split('/')[-1]))
        )[['id', 'title']]
    ], ignore_index=True)
    .drop_duplicates(['id'])
)

export_df = (
    relevant_labelled_df
    .copy()
    .drop('predictions', axis=1)
    .assign(url = [prepare_url(row.id, row.source) for index, row in relevant_labelled_df.iterrows()])
    .merge(
        titles_df,
        on='id',
        how='left'
    )
# )[['id', 'title', 'text', 'year', 'source', 'url', 'Detection', 'Management', 'Detection_', 'Management_']]
)[['id', 'title', 'text', 'year', 'source', 'url', 'topic', 'topic_name', 'prob_relevant', 'prediction']]


In [ ]:
detection_df = (
    pd.read_csv(ENRICHED_DATA_DIR / "openalex_patents_detection_labels.csv")
    .rename(columns ={
        "Detection": "prob_detection",
        "Management": "prob_management",
        "Detection_": "Detection",
        "Management_": "Management"
    })
    .assign(id = lambda df: df['id'].apply(lambda x: x.split('/')[-1]))
)

In [ ]:
topics_df = (
    export_df
    .query("prediction >= 0.8")
    .groupby(["id"])
    .agg(
        topic=("topic", list),
        topic_name=("topic_name", list),
    )
)
export_df = (
    export_df[['id', 'title', 'text', 'year', 'source', 'url']]
    .drop_duplicates("id")
    .merge(
        topics_df,
        on="id",
        how="left"
    )
    .merge(
        detection_df[["id", "prob_detection", "prob_management", "Detection", "Management"]],
        on="id",
        how="left"
    )
)

In [ ]:
# export_df.to_csv(PROJECT_DIR / 'outputs/enrichments/relevant_documents.csv', index=False)

In [ ]:
# remove altair row limit
alt.data_transformers.disable_max_rows()

# plot scatter with detection and management values
fig = (
    alt.Chart(
        export_df,
        width=500)
    .mark_point(opacity=0.2)
    .encode(
        x='Detection:Q',
        y='Management:Q',
        color='source',
        tooltip=['id', 'title', 'text', 'year', 'url']
    )
    .properties(
        title='Detection and Management scores'
    )
    .interactive()
)


In [ ]:
export_df.to_csv('relevant_documents.csv', index=False)

In [ ]:
export_df.info()

In [ ]:
ensemble_dict = {
    0: "0/5",
    0.2: "1/5",
    0.4: "2/5",
    0.6: "3/5",
    0.8: "4/5",
    1: "5/5"
}

topic = "mobile"
topic_df_ = (
    export_df[export_df.topic.astype(str).str.contains(topic)]
    .drop(["topic", "topic_name"], axis=1)
    .merge(relevant_labelled_df[['id', 'prediction', 'prob_relevant', 'topic', 'topic_name']].query("topic == @topic"),on='id', how='left') 
    .assign(ensemble = lambda df: df['prediction'].apply(lambda x: ensemble_dict[x]))
    .drop('prediction', axis=1)
)
topic_df_.to_csv(f"relevant_documents_{topic}.csv", index=False)

In [ ]:
topic_df_